In [1]:
import pyomo.environ as pyo

# Use the HiGHS solver through Pyomo's APPSI interface
SOLVER = pyo.SolverFactory('appsi_highs')
assert SOLVER.available(), "HiGHS (appsi_highs) solver is not available."
print("Solver ready:", SOLVER)



Solver ready: <pyomo.contrib.appsi.base.SolverFactoryClass.register.<locals>.decorator.<locals>.LegacySolver object at 0x0000023176649160>


In [2]:
model = pyo.ConcreteModel()

model.I = pyo.RangeSet(1,3)
model.J = pyo.RangeSet(1,3)

model.X = pyo.Var(model.I, model.J, domain=pyo.NonNegativeReals)

R = {
    (1,1) : 21, (1,2) : 21,   (1,3): 22,
    (2,1) : 26, (2,2) : 27,   (2,3): 28,
    (3,1) : 24, (3,2) : 25,   (3,3): 26
}
model.revenue = pyo.Param(model.I, model.J, initialize=R)

model.cases = pyo.Param(model.I, initialize={1: 800, 2: 400, 3: 500})

In [3]:
model.obj = pyo.Objective(expr=sum(model.X[i,j] * model.revenue[i,j] for i in model.I for j in model.J)
                          - 2 * sum(model.X[i, 1] for i in model.I)
                          - 0.1 * sum(model.X[i,2] * model.revenue[i,2] for i in model.I),
                          sense=pyo.maximize)

def cases_constraints(m, i):
    return sum(m.X[i,j] for j in m.J) <= m.cases[i]

model.cases_con = pyo.Constraint(model.I, rule=cases_constraints)

model.farmers_con = pyo.Constraint(expr = sum(model.X[i,3] for i in model.I) <= 80)

model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

In [4]:
SOLVER.solve(model)
print("Optimal value:", pyo.value(model.obj))
print("Optimal solution:")
for i in model.I:
    for j in model.J:
        print(f"X[{i},{j}] = {pyo.value(model.X[i,j])}")
        
print("\nDual values for case constraints:")
for i in model.I:
    print(f"Dual value for cases constraint {i}: {model.dual[model.cases_con[i]]}")
print(f"Dual value for farmers constraint: {model.dual[model.farmers_con]}")


Optimal value: 36466.0
Optimal solution:
X[1,1] = 800.0
X[1,2] = 0.0
X[1,3] = 0.0
X[2,1] = 0.0
X[2,2] = 320.0
X[2,3] = 80.0
X[3,1] = 0.0
X[3,2] = 500.0
X[3,3] = 0.0

Dual values for case constraints:
Dual value for cases constraint 1: 19.0
Dual value for cases constraint 2: 24.3
Dual value for cases constraint 3: 22.5
Dual value for farmers constraint: 3.6999999999999993
